In [42]:
import sys
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
from collections import Counter

# =========================================================
# CONFIGURATION GLOBALE
# =========================================================
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

TRAIN_FILE            = "client_1_train.csv"
TEST_FILE             = "test_1.csv"
ORIG_TRAIN_FILE       = "UNSW_NB15_training-set.csv"
ORIG_TEST_FILE        = "UNSW_NB15_testing-set.csv"

BATCH_SIZE            = 512
EPOCHS                = 10
LR                    = 1e-3
GAMMA_FOCAL           = 2.0   # γ=2.0 standard — stable et efficace
DROPOUT_RATE          = 0.3
NUM_CLASSES           = 10
MIN_SAMPLES_PER_CLASS = 3000  # seuil oversampling par classe présente


# =========================================================
# 1. FOCAL LOSS — VERSION NUMÉRIQUEMENT STABLE
# =========================================================
class MulticlassFocalLoss(nn.Module):
    """
    pt calculé directement depuis softmax() — jamais via exp(-CE).
    Évite le underflow (loss=0) quand le modèle est très confiant dès le début.
    """
    def __init__(self, weights=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.weights   = weights
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Probabilité softmax pour la vraie classe
        probs = F.softmax(inputs, dim=1)                          # (B, C)
        pt    = probs.gather(1, targets.unsqueeze(1)).squeeze(1)  # (B,)
        pt    = pt.clamp(min=1e-7, max=1.0 - 1e-7)               # stabilité numérique

        # CE standard (avec poids si fournis)
        ce    = F.cross_entropy(inputs, targets,
                                weight=self.weights, reduction='none')

        # Facteur focal : (1-pt)^gamma supprime le gradient des exemples faciles
        focal = (1.0 - pt) ** self.gamma * ce

        if self.reduction == 'mean':
            return focal.mean()
        elif self.reduction == 'sum':
            return focal.sum()
        return focal


# =========================================================
# 2. ARCHITECTURE MLP — BatchNorm + Dropout
# =========================================================
class IDS_MLP(nn.Module):
    def __init__(self, input_size, num_classes=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# =========================================================
# 3. OVERSAMPLING SYNTHÉTIQUE (SMOTE-like) — classes présentes uniquement
# =========================================================
def oversample_minority_classes(X, y, min_samples=MIN_SAMPLES_PER_CLASS,
                                noise_std=0.02):
    counts   = Counter(y.tolist())
    X_aug, y_aug = [X], [y]

    for cls, cnt in sorted(counts.items()):
        if cnt < min_samples:
            needed = min_samples - cnt
            idx    = np.where(y == cls)[0]
            samp   = np.random.choice(idx, size=needed, replace=True)
            X_new  = X[samp].copy()
            std    = X[idx].std(axis=0, keepdims=True).clip(min=1e-6)
            X_new += np.random.normal(0, noise_std, X_new.shape) * std
            X_aug.append(X_new)
            y_aug.append(np.full(needed, cls, dtype=y.dtype))
            print(f"  [OVERSAMPLE] classe {cls}: {cnt} → {cnt + needed}")

    return np.vstack(X_aug), np.concatenate(y_aug)


# =========================================================
# 4. PIPELINE PRINCIPAL
# =========================================================
class IDSPipeline:

    def __init__(self):
        self.scaler       = StandardScaler()
        self.le           = LabelEncoder()
        self.target_names = None
        self.model        = None
        self.loss_fn      = None
        self.optimizer    = None
        self.scheduler    = None
        self.train_loader = None
        self.test_loader  = None

    def _build_attack_mapping(self):
        orig_train = pd.read_csv(ORIG_TRAIN_FILE)
        orig_test  = pd.read_csv(ORIG_TEST_FILE)
        for df in (orig_train, orig_test):
            df['attack_cat'] = df['attack_cat'].str.strip()
        combined = pd.concat([orig_train['attack_cat'],
                               orig_test['attack_cat']]).astype(str)
        self.le.fit(combined)
        self.target_names = list(self.le.classes_)
        print(f"[INFO] Classes : {self.target_names}")

    def load_data(self, train_path, test_path):
        print(f"\n[LOAD] {train_path}  |  {test_path}")
        train = pd.read_csv(train_path)
        test  = pd.read_csv(test_path)

        col   = 'target' if 'target' in train.columns else 'label'
        X_tr  = train.drop(columns=[col]).values.astype(np.float32)
        y_tr  = train[col].values.astype(np.int64)
        X_te  = test.drop(columns=[col]).values.astype(np.float32)
        y_te  = test[col].values.astype(np.int64)

        X_tr  = self.scaler.fit_transform(X_tr).astype(np.float32)
        X_te  = self.scaler.transform(X_te).astype(np.float32)

        print("\n[OVERSAMPLE] Classes présentes dans ce client :")
        X_tr, y_tr = oversample_minority_classes(X_tr, y_tr)

        print("\n[DISTRIBUTION] Après oversampling :")
        for cls, cnt in sorted(Counter(y_tr.tolist()).items()):
            name = self.target_names[cls] if self.target_names else str(cls)
            print(f"  {name:<20} : {cnt:6d}")

        # Poids inverses (sqrt) sur les classes présentes uniquement
        counts_arr   = np.bincount(y_tr, minlength=NUM_CLASSES).astype(np.float32)
        counts_arr   = np.where(counts_arr == 0, 1e9, counts_arr)  # classes absentes → poids ~0
        w            = 1.0 / np.sqrt(counts_arr)
        w[counts_arr > 1e8] = 0.0                                   # vraiment absentes → 0
        class_weights = torch.tensor(w, dtype=torch.float32)

        # WeightedRandomSampler pour batches équilibrés
        sample_w = torch.tensor([w[y] for y in y_tr], dtype=torch.float32)
        sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w),
                                         replacement=True)

        train_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                                  torch.tensor(y_tr, dtype=torch.long))
        test_ds  = TensorDataset(torch.tensor(X_te, dtype=torch.float32),
                                  torch.tensor(y_te, dtype=torch.long))

        self.train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                                        sampler=sampler)
        self.test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                                        shuffle=False)

        print(f"\n[INFO] input_size={X_tr.shape[1]} | "
              f"train={len(X_tr)} | test={len(X_te)}")
        return class_weights, X_tr.shape[1]

    def build_model(self, input_size, class_weights):
        self.model     = IDS_MLP(input_size, NUM_CLASSES, DROPOUT_RATE)
        self.loss_fn   = MulticlassFocalLoss(weights=class_weights,
                                              gamma=GAMMA_FOCAL)
        self.optimizer = optim.AdamW(self.model.parameters(),
                                      lr=LR, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=4
        )
        n_params = sum(p.numel() for p in self.model.parameters())
        print(f"[MODEL] Paramètres : {n_params:,}")

    def train(self):
        print(f"\n{'='*65}")
        print(f" ENTRAÎNEMENT — {EPOCHS} ÉPOQUES | γ={GAMMA_FOCAL}")
        print(f"{'='*65}")

        for epoch in range(1, EPOCHS + 1):
            self.model.train()
            total_loss, n = 0.0, 0

            for X, y in self.train_loader:
                self.optimizer.zero_grad()
                loss = self.loss_fn(self.model(X), y)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                total_loss += loss.item() * X.size(0)
                n          += X.size(0)

            avg = total_loss / n
            self.scheduler.step(avg)

            if epoch % 5 == 0 or epoch == 1:
                lr_now = self.optimizer.param_groups[0]['lr']
                print(f"  Époque {epoch:3d}/{EPOCHS} | "
                      f"Focal Loss: {avg:.5f} | lr: {lr_now:.2e}")

        print("[FIN] Entraînement terminé.\n")

    def evaluate(self):
        print(f"\n{'='*75}")
        print(" ÉVALUATION — MÉTRIQUES PAR CLASSE")
        print(f"{'='*75}")

        self.model.eval()
        all_preds, all_targets = [], []

        with torch.no_grad():
            for X, y in self.test_loader:
                preds = torch.argmax(self.model(X), dim=1)
                all_preds.extend(preds.numpy())
                all_targets.extend(y.numpy())

        all_preds   = np.array(all_preds)
        all_targets = np.array(all_targets)

        unique = sorted(np.unique(np.concatenate([all_targets,
                                                   all_preds])).tolist())
        names  = [self.target_names[i] if self.target_names else str(i)
                  for i in unique]

        report = classification_report(
            all_targets, all_preds,
            labels=unique, target_names=names,
            output_dict=True, zero_division=0
        )

        acc     = accuracy_score(all_targets, all_preds)
        f1_mac  = f1_score(all_targets, all_preds, average='macro',  zero_division=0)
        f1_w    = f1_score(all_targets, all_preds, average='weighted', zero_division=0)

        print(f"\n  Accuracy        : {acc:.4f}")
        print(f"  F1 weighted     : {f1_w:.4f}")
        print(f"  F1 macro        : {f1_mac:.4f}  ← classes minoritaires")

        print(f"\n{'─'*75}")
        print(f"  {'Attaque':<20} {'Précision':>10} {'Rappel':>10} "
              f"{'F1':>10} {'Support':>10}")
        print(f"{'─'*75}")

        for name in names:
            m    = report[name]
            flag = " ⚠" if m['recall'] < 0.50 else ""
            print(f"  {name:<20} {m['precision']:>10.4f} {m['recall']:>10.4f} "
                  f"{m['f1-score']:>10.4f} {int(m['support']):>10}{flag}")

        print(f"{'─'*75}")
        wa = report['weighted avg']
        ma = report['macro avg']
        print(f"  {'weighted avg':<20} {wa['precision']:>10.4f} "
              f"{wa['recall']:>10.4f} {wa['f1-score']:>10.4f}")
        print(f"  {'macro avg':<20} {ma['precision']:>10.4f} "
              f"{ma['recall']:>10.4f} {ma['f1-score']:>10.4f}")
        print(f"{'='*75}\n")

        low = [(n, report[n]['recall']) for n in names if report[n]['recall'] < 0.50]
        if low:
            print(" Classes avec recall < 0.50 :")
            for n, r in low:
                print(f"  ⚠  {n:<20} recall={r:.4f}")
        else:
            print(" ✓ Toutes les classes ont recall ≥ 0.50")


# =========================================================
# POINT D'ENTRÉE
# =========================================================
if __name__ == "__main__":
    pipeline = IDSPipeline()
    pipeline._build_attack_mapping()
    class_weights, input_size = pipeline.load_data(TRAIN_FILE, TEST_FILE)
    pipeline.build_model(input_size, class_weights)
    pipeline.train()
    pipeline.evaluate()

[INFO] Classes : ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']

[LOAD] client_1_train.csv  |  test_1.csv

[OVERSAMPLE] Classes présentes dans ce client :
  [OVERSAMPLE] classe 1: 1746 → 3000
  [OVERSAMPLE] classe 9: 130 → 3000

[DISTRIBUTION] Après oversampling :
  Backdoor             :   3000
  Generic              :  40000
  Normal               :  18666
  Worms                :   3000

[INFO] input_size=42 | train=64666 | test=56498
[MODEL] Paramètres : 53,706

 ENTRAÎNEMENT — 10 ÉPOQUES | γ=2.0
  Époque   1/10 | Focal Loss: 0.00250 | lr: 1.00e-03
  Époque   5/10 | Focal Loss: 0.00044 | lr: 1.00e-03
  Époque  10/10 | Focal Loss: 0.00030 | lr: 1.00e-03
[FIN] Entraînement terminé.


 ÉVALUATION — MÉTRIQUES PAR CLASSE

  Accuracy        : 0.8875
  F1 weighted     : 0.9289
  F1 macro        : 0.5359  ← classes minoritaires

───────────────────────────────────────────────────────────────────────────
  Attaque         

In [43]:
import sys
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
from collections import Counter

# =========================================================
# CONFIGURATION GLOBALE
# =========================================================
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

TRAIN_FILE            = "client_2_train.csv"
TEST_FILE             = "test_2.csv"
ORIG_TRAIN_FILE       = "UNSW_NB15_training-set.csv"
ORIG_TEST_FILE        = "UNSW_NB15_testing-set.csv"

BATCH_SIZE            = 512
EPOCHS                = 10
LR                    = 1e-3
GAMMA_FOCAL           = 2.0   # γ=2.0 standard — stable et efficace
DROPOUT_RATE          = 0.3
NUM_CLASSES           = 10
MIN_SAMPLES_PER_CLASS = 3000  # seuil oversampling par classe présente


# =========================================================
# 1. FOCAL LOSS — VERSION NUMÉRIQUEMENT STABLE
# =========================================================
class MulticlassFocalLoss(nn.Module):
    """
    pt calculé directement depuis softmax() — jamais via exp(-CE).
    Évite le underflow (loss=0) quand le modèle est très confiant dès le début.
    """
    def __init__(self, weights=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.weights   = weights
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Probabilité softmax pour la vraie classe
        probs = F.softmax(inputs, dim=1)                          # (B, C)
        pt    = probs.gather(1, targets.unsqueeze(1)).squeeze(1)  # (B,)
        pt    = pt.clamp(min=1e-7, max=1.0 - 1e-7)               # stabilité numérique

        # CE standard (avec poids si fournis)
        ce    = F.cross_entropy(inputs, targets,
                                weight=self.weights, reduction='none')

        # Facteur focal : (1-pt)^gamma supprime le gradient des exemples faciles
        focal = (1.0 - pt) ** self.gamma * ce

        if self.reduction == 'mean':
            return focal.mean()
        elif self.reduction == 'sum':
            return focal.sum()
        return focal


# =========================================================
# 2. ARCHITECTURE MLP — BatchNorm + Dropout
# =========================================================
class IDS_MLP(nn.Module):
    def __init__(self, input_size, num_classes=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# =========================================================
# 3. OVERSAMPLING SYNTHÉTIQUE (SMOTE-like) — classes présentes uniquement
# =========================================================
def oversample_minority_classes(X, y, min_samples=MIN_SAMPLES_PER_CLASS,
                                noise_std=0.02):
    counts   = Counter(y.tolist())
    X_aug, y_aug = [X], [y]

    for cls, cnt in sorted(counts.items()):
        if cnt < min_samples:
            needed = min_samples - cnt
            idx    = np.where(y == cls)[0]
            samp   = np.random.choice(idx, size=needed, replace=True)
            X_new  = X[samp].copy()
            std    = X[idx].std(axis=0, keepdims=True).clip(min=1e-6)
            X_new += np.random.normal(0, noise_std, X_new.shape) * std
            X_aug.append(X_new)
            y_aug.append(np.full(needed, cls, dtype=y.dtype))
            print(f"  [OVERSAMPLE] classe {cls}: {cnt} → {cnt + needed}")

    return np.vstack(X_aug), np.concatenate(y_aug)


# =========================================================
# 4. PIPELINE PRINCIPAL
# =========================================================
class IDSPipeline:

    def __init__(self):
        self.scaler       = StandardScaler()
        self.le           = LabelEncoder()
        self.target_names = None
        self.model        = None
        self.loss_fn      = None
        self.optimizer    = None
        self.scheduler    = None
        self.train_loader = None
        self.test_loader  = None

    def _build_attack_mapping(self):
        orig_train = pd.read_csv(ORIG_TRAIN_FILE)
        orig_test  = pd.read_csv(ORIG_TEST_FILE)
        for df in (orig_train, orig_test):
            df['attack_cat'] = df['attack_cat'].str.strip()
        combined = pd.concat([orig_train['attack_cat'],
                               orig_test['attack_cat']]).astype(str)
        self.le.fit(combined)
        self.target_names = list(self.le.classes_)
        print(f"[INFO] Classes : {self.target_names}")

    def load_data(self, train_path, test_path):
        print(f"\n[LOAD] {train_path}  |  {test_path}")
        train = pd.read_csv(train_path)
        test  = pd.read_csv(test_path)

        col   = 'target' if 'target' in train.columns else 'label'
        X_tr  = train.drop(columns=[col]).values.astype(np.float32)
        y_tr  = train[col].values.astype(np.int64)
        X_te  = test.drop(columns=[col]).values.astype(np.float32)
        y_te  = test[col].values.astype(np.int64)

        X_tr  = self.scaler.fit_transform(X_tr).astype(np.float32)
        X_te  = self.scaler.transform(X_te).astype(np.float32)

        print("\n[OVERSAMPLE] Classes présentes dans ce client :")
        X_tr, y_tr = oversample_minority_classes(X_tr, y_tr)

        print("\n[DISTRIBUTION] Après oversampling :")
        for cls, cnt in sorted(Counter(y_tr.tolist()).items()):
            name = self.target_names[cls] if self.target_names else str(cls)
            print(f"  {name:<20} : {cnt:6d}")

        # Poids inverses (sqrt) sur les classes présentes uniquement
        counts_arr   = np.bincount(y_tr, minlength=NUM_CLASSES).astype(np.float32)
        counts_arr   = np.where(counts_arr == 0, 1e9, counts_arr)  # classes absentes → poids ~0
        w            = 1.0 / np.sqrt(counts_arr)
        w[counts_arr > 1e8] = 0.0                                   # vraiment absentes → 0
        class_weights = torch.tensor(w, dtype=torch.float32)

        # WeightedRandomSampler pour batches équilibrés
        sample_w = torch.tensor([w[y] for y in y_tr], dtype=torch.float32)
        sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w),
                                         replacement=True)

        train_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                                  torch.tensor(y_tr, dtype=torch.long))
        test_ds  = TensorDataset(torch.tensor(X_te, dtype=torch.float32),
                                  torch.tensor(y_te, dtype=torch.long))

        self.train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                                        sampler=sampler)
        self.test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                                        shuffle=False)

        print(f"\n[INFO] input_size={X_tr.shape[1]} | "
              f"train={len(X_tr)} | test={len(X_te)}")
        return class_weights, X_tr.shape[1]

    def build_model(self, input_size, class_weights):
        self.model     = IDS_MLP(input_size, NUM_CLASSES, DROPOUT_RATE)
        self.loss_fn   = MulticlassFocalLoss(weights=class_weights,
                                              gamma=GAMMA_FOCAL)
        self.optimizer = optim.AdamW(self.model.parameters(),
                                      lr=LR, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=4
        )
        n_params = sum(p.numel() for p in self.model.parameters())
        print(f"[MODEL] Paramètres : {n_params:,}")

    def train(self):
        print(f"\n{'='*65}")
        print(f" ENTRAÎNEMENT — {EPOCHS} ÉPOQUES | γ={GAMMA_FOCAL}")
        print(f"{'='*65}")

        for epoch in range(1, EPOCHS + 1):
            self.model.train()
            total_loss, n = 0.0, 0

            for X, y in self.train_loader:
                self.optimizer.zero_grad()
                loss = self.loss_fn(self.model(X), y)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                total_loss += loss.item() * X.size(0)
                n          += X.size(0)

            avg = total_loss / n
            self.scheduler.step(avg)

            if epoch % 5 == 0 or epoch == 1:
                lr_now = self.optimizer.param_groups[0]['lr']
                print(f"  Époque {epoch:3d}/{EPOCHS} | "
                      f"Focal Loss: {avg:.5f} | lr: {lr_now:.2e}")

        print("[FIN] Entraînement terminé.\n")

    def evaluate(self):
        print(f"\n{'='*75}")
        print(" ÉVALUATION — MÉTRIQUES PAR CLASSE")
        print(f"{'='*75}")

        self.model.eval()
        all_preds, all_targets = [], []

        with torch.no_grad():
            for X, y in self.test_loader:
                preds = torch.argmax(self.model(X), dim=1)
                all_preds.extend(preds.numpy())
                all_targets.extend(y.numpy())

        all_preds   = np.array(all_preds)
        all_targets = np.array(all_targets)

        unique = sorted(np.unique(np.concatenate([all_targets,
                                                   all_preds])).tolist())
        names  = [self.target_names[i] if self.target_names else str(i)
                  for i in unique]

        report = classification_report(
            all_targets, all_preds,
            labels=unique, target_names=names,
            output_dict=True, zero_division=0
        )

        acc     = accuracy_score(all_targets, all_preds)
        f1_mac  = f1_score(all_targets, all_preds, average='macro',  zero_division=0)
        f1_w    = f1_score(all_targets, all_preds, average='weighted', zero_division=0)

        print(f"\n  Accuracy        : {acc:.4f}")
        print(f"  F1 weighted     : {f1_w:.4f}")
        print(f"  F1 macro        : {f1_mac:.4f}  ← classes minoritaires")

        print(f"\n{'─'*75}")
        print(f"  {'Attaque':<20} {'Précision':>10} {'Rappel':>10} "
              f"{'F1':>10} {'Support':>10}")
        print(f"{'─'*75}")

        for name in names:
            m    = report[name]
            flag = " ⚠" if m['recall'] < 0.50 else ""
            print(f"  {name:<20} {m['precision']:>10.4f} {m['recall']:>10.4f} "
                  f"{m['f1-score']:>10.4f} {int(m['support']):>10}{flag}")

        print(f"{'─'*75}")
        wa = report['weighted avg']
        ma = report['macro avg']
        print(f"  {'weighted avg':<20} {wa['precision']:>10.4f} "
              f"{wa['recall']:>10.4f} {wa['f1-score']:>10.4f}")
        print(f"  {'macro avg':<20} {ma['precision']:>10.4f} "
              f"{ma['recall']:>10.4f} {ma['f1-score']:>10.4f}")
        print(f"{'='*75}\n")

        low = [(n, report[n]['recall']) for n in names if report[n]['recall'] < 0.50]
        if low:
            print(" Classes avec recall < 0.50 :")
            for n, r in low:
                print(f"  ⚠  {n:<20} recall={r:.4f}")
        else:
            print(" ✓ Toutes les classes ont recall ≥ 0.50")


# =========================================================
# POINT D'ENTRÉE
# =========================================================
if __name__ == "__main__":
    pipeline = IDSPipeline()
    pipeline._build_attack_mapping()
    class_weights, input_size = pipeline.load_data(TRAIN_FILE, TEST_FILE)
    pipeline.build_model(input_size, class_weights)
    pipeline.train()
    pipeline.evaluate()

[INFO] Classes : ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']

[LOAD] client_2_train.csv  |  test_2.csv

[OVERSAMPLE] Classes présentes dans ce client :
  [OVERSAMPLE] classe 0: 2000 → 3000
  [OVERSAMPLE] classe 8: 1133 → 3000

[DISTRIBUTION] Après oversampling :
  Analysis             :   3000
  Exploits             :  33393
  Normal               :  18668
  Shellcode            :   3000

[INFO] input_size=42 | train=58061 | test=49187
[MODEL] Paramètres : 53,706

 ENTRAÎNEMENT — 10 ÉPOQUES | γ=2.0
  Époque   1/10 | Focal Loss: 0.00492 | lr: 1.00e-03
  Époque   5/10 | Focal Loss: 0.00131 | lr: 1.00e-03
  Époque  10/10 | Focal Loss: 0.00111 | lr: 1.00e-03
[FIN] Entraînement terminé.


 ÉVALUATION — MÉTRIQUES PAR CLASSE

  Accuracy        : 0.7898
  F1 weighted     : 0.8498
  F1 macro        : 0.4777  ← classes minoritaires

───────────────────────────────────────────────────────────────────────────
  Attaque        

In [44]:
import sys
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
from collections import Counter

# =========================================================
# CONFIGURATION GLOBALE
# =========================================================
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

TRAIN_FILE            = "client_0_train.csv"
TEST_FILE             = "test_0.csv"
ORIG_TRAIN_FILE       = "UNSW_NB15_training-set.csv"
ORIG_TEST_FILE        = "UNSW_NB15_testing-set.csv"

BATCH_SIZE            = 512
EPOCHS                = 10
LR                    = 1e-3
GAMMA_FOCAL           = 2.0   # γ=2.0 standard — stable et efficace
DROPOUT_RATE          = 0.3
NUM_CLASSES           = 10
MIN_SAMPLES_PER_CLASS = 3000  # seuil oversampling par classe présente


# =========================================================
# 1. FOCAL LOSS — VERSION NUMÉRIQUEMENT STABLE
# =========================================================
class MulticlassFocalLoss(nn.Module):
    """
    pt calculé directement depuis softmax() — jamais via exp(-CE).
    Évite le underflow (loss=0) quand le modèle est très confiant dès le début.
    """
    def __init__(self, weights=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.weights   = weights
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Probabilité softmax pour la vraie classe
        probs = F.softmax(inputs, dim=1)                          # (B, C)
        pt    = probs.gather(1, targets.unsqueeze(1)).squeeze(1)  # (B,)
        pt    = pt.clamp(min=1e-7, max=1.0 - 1e-7)               # stabilité numérique

        # CE standard (avec poids si fournis)
        ce    = F.cross_entropy(inputs, targets,
                                weight=self.weights, reduction='none')

        # Facteur focal : (1-pt)^gamma supprime le gradient des exemples faciles
        focal = (1.0 - pt) ** self.gamma * ce

        if self.reduction == 'mean':
            return focal.mean()
        elif self.reduction == 'sum':
            return focal.sum()
        return focal


# =========================================================
# 2. ARCHITECTURE MLP — BatchNorm + Dropout
# =========================================================
class IDS_MLP(nn.Module):
    def __init__(self, input_size, num_classes=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# =========================================================
# 3. OVERSAMPLING SYNTHÉTIQUE (SMOTE-like) — classes présentes uniquement
# =========================================================
def oversample_minority_classes(X, y, min_samples=MIN_SAMPLES_PER_CLASS,
                                noise_std=0.02):
    counts   = Counter(y.tolist())
    X_aug, y_aug = [X], [y]

    for cls, cnt in sorted(counts.items()):
        if cnt < min_samples:
            needed = min_samples - cnt
            idx    = np.where(y == cls)[0]
            samp   = np.random.choice(idx, size=needed, replace=True)
            X_new  = X[samp].copy()
            std    = X[idx].std(axis=0, keepdims=True).clip(min=1e-6)
            X_new += np.random.normal(0, noise_std, X_new.shape) * std
            X_aug.append(X_new)
            y_aug.append(np.full(needed, cls, dtype=y.dtype))
            print(f"  [OVERSAMPLE] classe {cls}: {cnt} → {cnt + needed}")

    return np.vstack(X_aug), np.concatenate(y_aug)


# =========================================================
# 4. PIPELINE PRINCIPAL
# =========================================================
class IDSPipeline:

    def __init__(self):
        self.scaler       = StandardScaler()
        self.le           = LabelEncoder()
        self.target_names = None
        self.model        = None
        self.loss_fn      = None
        self.optimizer    = None
        self.scheduler    = None
        self.train_loader = None
        self.test_loader  = None

    def _build_attack_mapping(self):
        orig_train = pd.read_csv(ORIG_TRAIN_FILE)
        orig_test  = pd.read_csv(ORIG_TEST_FILE)
        for df in (orig_train, orig_test):
            df['attack_cat'] = df['attack_cat'].str.strip()
        combined = pd.concat([orig_train['attack_cat'],
                               orig_test['attack_cat']]).astype(str)
        self.le.fit(combined)
        self.target_names = list(self.le.classes_)
        print(f"[INFO] Classes : {self.target_names}")

    def load_data(self, train_path, test_path):
        print(f"\n[LOAD] {train_path}  |  {test_path}")
        train = pd.read_csv(train_path)
        test  = pd.read_csv(test_path)

        col   = 'target' if 'target' in train.columns else 'label'
        X_tr  = train.drop(columns=[col]).values.astype(np.float32)
        y_tr  = train[col].values.astype(np.int64)
        X_te  = test.drop(columns=[col]).values.astype(np.float32)
        y_te  = test[col].values.astype(np.int64)

        X_tr  = self.scaler.fit_transform(X_tr).astype(np.float32)
        X_te  = self.scaler.transform(X_te).astype(np.float32)

        print("\n[OVERSAMPLE] Classes présentes dans ce client :")
        X_tr, y_tr = oversample_minority_classes(X_tr, y_tr)

        print("\n[DISTRIBUTION] Après oversampling :")
        for cls, cnt in sorted(Counter(y_tr.tolist()).items()):
            name = self.target_names[cls] if self.target_names else str(cls)
            print(f"  {name:<20} : {cnt:6d}")

        # Poids inverses (sqrt) sur les classes présentes uniquement
        counts_arr   = np.bincount(y_tr, minlength=NUM_CLASSES).astype(np.float32)
        counts_arr   = np.where(counts_arr == 0, 1e9, counts_arr)  # classes absentes → poids ~0
        w            = 1.0 / np.sqrt(counts_arr)
        w[counts_arr > 1e8] = 0.0                                   # vraiment absentes → 0
        class_weights = torch.tensor(w, dtype=torch.float32)

        # WeightedRandomSampler pour batches équilibrés
        sample_w = torch.tensor([w[y] for y in y_tr], dtype=torch.float32)
        sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w),
                                         replacement=True)

        train_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                                  torch.tensor(y_tr, dtype=torch.long))
        test_ds  = TensorDataset(torch.tensor(X_te, dtype=torch.float32),
                                  torch.tensor(y_te, dtype=torch.long))

        self.train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                                        sampler=sampler)
        self.test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                                        shuffle=False)

        print(f"\n[INFO] input_size={X_tr.shape[1]} | "
              f"train={len(X_tr)} | test={len(X_te)}")
        return class_weights, X_tr.shape[1]

    def build_model(self, input_size, class_weights):
        self.model     = IDS_MLP(input_size, NUM_CLASSES, DROPOUT_RATE)
        self.loss_fn   = MulticlassFocalLoss(weights=class_weights,
                                              gamma=GAMMA_FOCAL)
        self.optimizer = optim.AdamW(self.model.parameters(),
                                      lr=LR, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=4
        )
        n_params = sum(p.numel() for p in self.model.parameters())
        print(f"[MODEL] Paramètres : {n_params:,}")

    def train(self):
        print(f"\n{'='*65}")
        print(f" ENTRAÎNEMENT — {EPOCHS} ÉPOQUES | γ={GAMMA_FOCAL}")
        print(f"{'='*65}")

        for epoch in range(1, EPOCHS + 1):
            self.model.train()
            total_loss, n = 0.0, 0

            for X, y in self.train_loader:
                self.optimizer.zero_grad()
                loss = self.loss_fn(self.model(X), y)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                total_loss += loss.item() * X.size(0)
                n          += X.size(0)

            avg = total_loss / n
            self.scheduler.step(avg)

            if epoch % 5 == 0 or epoch == 1:
                lr_now = self.optimizer.param_groups[0]['lr']
                print(f"  Époque {epoch:3d}/{EPOCHS} | "
                      f"Focal Loss: {avg:.5f} | lr: {lr_now:.2e}")

        print("[FIN] Entraînement terminé.\n")

    def evaluate(self):
        print(f"\n{'='*75}")
        print(" ÉVALUATION — MÉTRIQUES PAR CLASSE")
        print(f"{'='*75}")

        self.model.eval()
        all_preds, all_targets = [], []

        with torch.no_grad():
            for X, y in self.test_loader:
                preds = torch.argmax(self.model(X), dim=1)
                all_preds.extend(preds.numpy())
                all_targets.extend(y.numpy())

        all_preds   = np.array(all_preds)
        all_targets = np.array(all_targets)

        unique = sorted(np.unique(np.concatenate([all_targets,
                                                   all_preds])).tolist())
        names  = [self.target_names[i] if self.target_names else str(i)
                  for i in unique]

        report = classification_report(
            all_targets, all_preds,
            labels=unique, target_names=names,
            output_dict=True, zero_division=0
        )

        acc     = accuracy_score(all_targets, all_preds)
        f1_mac  = f1_score(all_targets, all_preds, average='macro',  zero_division=0)
        f1_w    = f1_score(all_targets, all_preds, average='weighted', zero_division=0)

        print(f"\n  Accuracy        : {acc:.4f}")
        print(f"  F1 weighted     : {f1_w:.4f}")
        print(f"  F1 macro        : {f1_mac:.4f}  ← classes minoritaires")

        print(f"\n{'─'*75}")
        print(f"  {'Attaque':<20} {'Précision':>10} {'Rappel':>10} "
              f"{'F1':>10} {'Support':>10}")
        print(f"{'─'*75}")

        for name in names:
            m    = report[name]
            flag = " ⚠" if m['recall'] < 0.50 else ""
            print(f"  {name:<20} {m['precision']:>10.4f} {m['recall']:>10.4f} "
                  f"{m['f1-score']:>10.4f} {int(m['support']):>10}{flag}")

        print(f"{'─'*75}")
        wa = report['weighted avg']
        ma = report['macro avg']
        print(f"  {'weighted avg':<20} {wa['precision']:>10.4f} "
              f"{wa['recall']:>10.4f} {wa['f1-score']:>10.4f}")
        print(f"  {'macro avg':<20} {ma['precision']:>10.4f} "
              f"{ma['recall']:>10.4f} {ma['f1-score']:>10.4f}")
        print(f"{'='*75}\n")

        low = [(n, report[n]['recall']) for n in names if report[n]['recall'] < 0.50]
        if low:
            print(" Classes avec recall < 0.50 :")
            for n, r in low:
                print(f"  ⚠  {n:<20} recall={r:.4f}")
        else:
            print(" ✓ Toutes les classes ont recall ≥ 0.50")


# =========================================================
# POINT D'ENTRÉE
# =========================================================
if __name__ == "__main__":
    pipeline = IDSPipeline()
    pipeline._build_attack_mapping()
    class_weights, input_size = pipeline.load_data(TRAIN_FILE, TEST_FILE)
    pipeline.build_model(input_size, class_weights)
    pipeline.train()
    pipeline.evaluate()

[INFO] Classes : ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']

[LOAD] client_0_train.csv  |  test_0.csv

[OVERSAMPLE] Classes présentes dans ce client :

[DISTRIBUTION] Après oversampling :
  DoS                  :  12264
  Fuzzers              :  18184
  Normal               :  18666
  Reconnaissance       :  10491

[INFO] input_size=42 | train=59605 | test=50647
[MODEL] Paramètres : 53,706

 ENTRAÎNEMENT — 10 ÉPOQUES | γ=2.0
  Époque   1/10 | Focal Loss: 0.00500 | lr: 1.00e-03
  Époque   5/10 | Focal Loss: 0.00191 | lr: 1.00e-03
  Époque  10/10 | Focal Loss: 0.00165 | lr: 1.00e-03
[FIN] Entraînement terminé.


 ÉVALUATION — MÉTRIQUES PAR CLASSE

  Accuracy        : 0.6608
  F1 weighted     : 0.7027
  F1 macro        : 0.6383  ← classes minoritaires

───────────────────────────────────────────────────────────────────────────
  Attaque               Précision     Rappel         F1    Support
────────────────────────